# 25 — Policy gradient: REINFORCE and the PPO clip

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omarahmedZa/ai-ml-course/blob/main/notebooks/025-policy-gradient-at-scale.ipynb)

Instead of learning values first, policy-gradient methods turn the policy itself into the thing you nudge: good outcomes make the action taken more likely, bad ones make it rarer. You will run REINFORCE on a two-armed bandit, watch the policy move, and then look at the clip that PPO puts on the size of an update.

## 1. The numbers: one parameter, two arms

The policy is a single number `p`, the probability of pulling arm 1, and each arm pays 1 with a fixed chance. With `p = 0.5` the expected reward is the plain average of the two arms. The last line shows how the two log-probability gradients cancel, because probabilities must sum to one.

In [ ]:
import numpy as np                                   # the array library
p = 0.5                                              # the current probability of pulling arm 1
means = np.array([0.35, 0.65])                       # true chance of a reward on each arm
expected = np.array([1 - p, p]) @ means              # value of the current policy
print("expected reward:", round(expected, 3))        # 0.5 * 0.35 + 0.5 * 0.65
print("best arm pays:", means.max())                 # 0.65
grad_log = np.array([-p, 1 - p])                     # d log pi / d theta for each action
print("grad of log pi:", grad_log)                   # arm 1 gets +0.5, arm 0 gets -0.5
print("they sum to:", grad_log.sum())                # 0.0: the two gradients cancel

<details>
<summary><strong>Line by line</strong> (click to open)</summary>

<ul><li><code>import numpy as np</code> — loads the array library.</li><li><code>p = 0.5</code> — the single number that is the whole policy.</li><li><code>means = ...</code> — the true reward rate of each arm.</li><li><code>expected = ...</code> — probability of each action times its reward rate.</li><li><code>print("expected reward:", ...)</code> — what the policy is worth right now.</li><li><code>print("best arm pays:", ...)</code> — the ceiling.</li><li><code>grad_log = ...</code> — how the log-probability of each action moves with the parameter.</li><li><code>print("grad of log pi:", ...)</code> — a push toward one arm is a pull away from the other.</li><li><code>print("they sum to:", ...)</code> — the gradients cancel.</li></ul>

</details>

## 2. Step by step: REINFORCE, one episode at a time

Each episode pulls one arm, sees a 0 or a 1, and compares that reward with a running average. The surprise, reward minus baseline, scales the push toward whatever arm was just pulled, so a lucky pull makes that arm more likely and an unlucky one makes it rarer.

In [ ]:
rng = np.random.default_rng(0)                       # fixed seed: the run is reproducible
theta = 0.0                                          # the parameter; p = sigmoid(theta)
baseline = 0.0                                       # running average reward
p_hist = []                                          # probability of arm 1 after each episode
for ep in range(400):                                # 400 episodes
    p = 1. / (1. + np.exp(-theta))                   # sigmoid: parameter to probability
    a = int(rng.random() > p)                        # pull arm 1 with probability p
    reward = float(rng.random() < means[a])          # a Bernoulli reward from that arm
    advantage = reward - baseline                    # better or worse than usual?
    theta = theta + 0.3 * advantage * (a - p)        # push toward the action just taken
    baseline = baseline + 0.05 * (reward - baseline) # track the average reward
    p_hist.append(1. / (1. + np.exp(-theta)))        # record the policy after the update
    if ep % 100 == 0:                                # four checkpoints
        print(f"episode {ep:3d}  p(arm 1) {p_hist[-1]:.3f}  baseline {baseline:.3f}")   # progress

<details>
<summary><strong>Line by line</strong> (click to open)</summary>

<ul><li><code>rng = np.random.default_rng(0)</code> — a seeded random source.</li><li><code>theta = 0.0</code> — the parameter behind the policy.</li><li><code>baseline = 0.0</code> — the running average that defines surprise.</li><li><code>p_hist = []</code> — the policy's history.</li><li><code>for ep in range(400):</code> — the episode loop.</li><li><code>p = 1. / (1. + np.exp(-theta))</code> — sigmoid turns any number into a probability.</li><li><code>a = int(rng.random() &gt; p)</code> — sample an action.</li><li><code>reward = float(...)</code> — sample the reward.</li><li><code>advantage = reward - baseline</code> — the surprise.</li><li><code>theta = theta + 0.3 * ...</code> — REINFORCE: step size times surprise times gradient.</li><li><code>baseline = ...</code> — a slow average of rewards.</li><li><code>p_hist.append(...)</code> — record the new policy.</li><li><code>if ep % 100 == 0:</code> — print at four checkpoints only.</li><li><code>print(...)</code> — policy and baseline at each checkpoint.</li></ul>

</details>

## 3. Plot: the policy climbing

Single episodes are noisy, so the light line is the raw policy and the dark one is a 25-episode average. The policy rises toward always-arming 1, but it approaches the top asymptotically: each step is proportional to how uncertain the policy still is.

In [ ]:
import matplotlib.pyplot as plt                      # plotting library
smooth = np.convolve(p_hist, np.ones(25) / 25, mode="valid")   # 25-episode moving average
plt.figure(figsize=(6, 3))                           # one wide panel
plt.plot(p_hist, color="tab:blue", alpha=0.35, label="p(arm 1) each episode")   # raw policy
plt.plot(range(24, len(p_hist)), smooth, color="tab:red", label="25-episode average")   # smoothed
plt.axhline(1.0, color="grey", linestyle="--", label="always pull arm 1")   # the limit
plt.ylim(0, 1.05)                                    # probability axis
plt.xlabel("episode")                                # x-axis label
plt.ylabel("probability of pulling arm 1")           # y-axis label
plt.legend(fontsize=8)                               # name the three lines
plt.title("the policy finds the arm worth 0.65 and closes in slowly")   # takeaway
plt.show()                                           # render

<details>
<summary><strong>Line by line</strong> (click to open)</summary>

<ul><li><code>import matplotlib.pyplot as plt</code> — loads the plotting library.</li><li><code>smooth = np.convolve(...)</code> — a 25-episode moving average of the policy.</li><li><code>plt.figure(...)</code> — one panel.</li><li><code>plt.plot(p_hist, ...)</code> — every episode's policy, faintly.</li><li><code>plt.plot(range(24, ...), smooth, ...)</code> — the average, aligned with its episodes.</li><li><code>plt.axhline(1.0, ...)</code> — the policy the learner is heading toward.</li><li><code>plt.ylim(0, 1.05)</code> — a probability axis.</li><li><code>plt.xlabel("episode")</code> — names the horizontal axis.</li><li><code>plt.ylabel(...)</code> — names the vertical axis.</li><li><code>plt.legend(...)</code> — name the lines.</li><li><code>plt.title(...)</code> — the sentence the figure is making.</li><li><code>plt.show()</code> — render.</li></ul>

</details>

## 4. Interactive: drag the step size

The slider sets the learning rate and redraws the policy over the same 400 episodes with the same seed, so only the step size changes. Small steps leave the policy near 0.5; large steps race to the top. The title prints the policy at the last episode.

In [ ]:
from ipywidgets import interact                       # sliders that rerun a function
def show(lr=0.1):                                    # lr comes from the slider
    rng = np.random.default_rng(0)                   # same seed every drag
    theta, base, hist = 0.0, 0.0, []                 # parameter, baseline, history
    for ep in range(400):                            # the same 400 episodes
        p = 1. / (1. + np.exp(-theta))               # current policy
        a = int(rng.random() > p)                    # sample an action
        reward = float(rng.random() < means[a])      # sample a reward
        theta = theta + lr * (reward - base) * (a - p)   # REINFORCE update, sized by the slider
        base = base + 0.05 * (reward - base)         # update the baseline
        hist.append(1. / (1. + np.exp(-theta)))      # record the policy
    plt.figure(figsize=(5, 3))                       # a fresh figure per drag
    plt.plot(hist, color="tab:green")                # policy against episode
    plt.ylim(0, 1.05)                                # fixed axis, so curves are comparable
    plt.xlabel("episode")                            # x-axis label
    plt.ylabel("p(arm 1)")                           # y-axis label
    plt.title(f"lr = {lr:.2f}: final p = {hist[-1]:.2f}")   # live readout
    plt.show()                                       # draw it
    print(f"lr {lr:.2f} -> final p {hist[-1]:.3f}")  # the same number as text
interact(show, lr=(0.01, 0.5, 0.01))                 # the slider

<details>
<summary><strong>Line by line</strong> (click to open)</summary>

<ul><li><code>from ipywidgets import interact</code> — the slider widget.</li><li><code>def show(lr=0.1):</code> — one function per slider position.</li><li><code>rng = np.random.default_rng(0)</code> — the same random draws for every setting.</li><li><code>theta, base, hist = ...</code> — start clean.</li><li><code>for ep in range(400):</code> — the episode loop.</li><li><code>p = ...</code> — current policy.</li><li><code>a = ...</code> — sample an action.</li><li><code>reward = ...</code> — sample a reward.</li><li><code>theta = theta + lr * ...</code> — the update, sized by the slider.</li><li><code>base = ...</code> — update the baseline.</li><li><code>hist.append(...)</code> — record the policy.</li><li><code>plt.figure(...)</code> — a new figure per drag.</li><li><code>plt.plot(hist, ...)</code> — the policy curve.</li><li><code>plt.ylim(0, 1.05)</code> — a fixed axis for comparison.</li><li><code>plt.xlabel("episode")</code> — names the horizontal axis.</li><li><code>plt.ylabel("p(arm 1)")</code> — names the vertical axis.</li><li><code>plt.title(...)</code> — the final policy, printed on the plot.</li><li><code>plt.show()</code> — draw it.</li><li><code>print(...)</code> — the same number as text.</li><li><code>interact(show, lr=(0.01, 0.5, 0.01))</code> — the slider, its range and its step.</li></ul>

</details>

## 5. When it breaks: the clip is silent inside the band

PPO multiplies the advantage by the ratio of new to old action probability, then clamps that ratio to the band 0.7 to 1.3. A ratio of 1.2 sits inside the band, so the objective pays the full 1.2*A and nothing is protected. Past 1.3 the objective is flat, so the gradient is exactly zero and the step is never undone.

In [ ]:
ratio = np.linspace(0.4, 2.0, 161)                   # how much the new policy grew: new / old
clip = np.clip(ratio, 0.7, 1.3)                      # PPO's trusted band
A = 1.0                                              # a positive advantage: this action paid off
objective = np.minimum(ratio * A, clip * A)          # the clipped surrogate for A > 0
neg = np.minimum(ratio * -A, clip * -A)              # the same rule for A < 0
print("ratio 1.2, A = +1 -> objective", np.minimum(1.2, 1.3) * A)   # 1.2: the clip never binds
print("ratio 1.5, A = +1 -> objective", np.minimum(1.5, 1.3) * A)   # 1.3: clamped, and flat after
plt.figure(figsize=(6, 3))                           # one panel
plt.plot(ratio, objective, label="A = +1")           # positive advantage
plt.plot(ratio, neg, label="A = -1")                 # negative advantage
plt.axvline(0.7, color="grey", linestyle="--")       # lower edge of the band
plt.axvline(1.3, color="grey", linestyle="--")       # upper edge of the band
plt.axhline(1.3, color="tab:red", linestyle=":", linewidth=1)   # where A = +1 stops paying
plt.plot([1.2], [1.2], "ko")                         # inside the band: paid in full
plt.plot([1.5], [1.3], "ks")                         # outside the band: frozen at 1.3
plt.xlabel("policy ratio, new / old")                # x-axis label
plt.ylabel("objective, ratio x advantage")           # y-axis label
plt.legend(fontsize=8)                               # name the two curves
plt.title("inside the band the clip is silent; outside it the slope is zero")   # takeaway
plt.show()                                           # render

<details>
<summary><strong>Line by line</strong> (click to open)</summary>

<ul><li><code>ratio = np.linspace(0.4, 2.0, 161)</code> — candidate ratios from shrunken to doubled.</li><li><code>clip = np.clip(ratio, 0.7, 1.3)</code> — the ratio after PPO's clamp.</li><li><code>A = 1.0</code> — the advantage, held positive.</li><li><code>objective = ...</code> — the smaller of clipped and unclipped, for A &gt; 0.</li><li><code>neg = ...</code> — the same formula with the sign flipped.</li><li><code>print("ratio 1.2 ...")</code> — 1.2 * A: the clip did nothing.</li><li><code>print("ratio 1.5 ...")</code> — 1.3 * A: clamped, and no longer changing.</li><li><code>plt.figure(...)</code> — one panel.</li><li><code>plt.plot(ratio, objective, ...)</code> — the positive-advantage objective.</li><li><code>plt.plot(ratio, neg, ...)</code> — the negative-advantage objective.</li><li><code>plt.axvline(0.7, ...)</code> — lower edge of the band.</li><li><code>plt.axvline(1.3, ...)</code> — upper edge.</li><li><code>plt.axhline(1.3, ...)</code> — the plateau for a positive advantage.</li><li><code>plt.plot([1.2], [1.2], "ko")</code> — the unclipped point.</li><li><code>plt.plot([1.5], [1.3], "ks")</code> — the clamped point.</li><li><code>plt.xlabel(...)</code> — names the horizontal axis.</li><li><code>plt.ylabel(...)</code> — names the vertical axis.</li><li><code>plt.legend(...)</code> — name the curves.</li><li><code>plt.title(...)</code> — the sentence the figure is making.</li><li><code>plt.show()</code> — render.</li></ul>

</details>

## Your turn

Set `means` to `[0.45, 0.5]` and predict how many episodes the policy needs before it looks decided. Then compute by hand what the section 5 objective pays at a ratio of 0.6 with `A = +1`, and check your answer against the plot.